# Lektion 08 - Multi-Agent Design Pattern


## Installation


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Varför Multi-Agent System?

Verkliga uppgifter som reseplanering involverar många olika typer av expertis — logistik, lokal kunskap, budgetering och mer. En enda agent som försöker hantera allt blir snabbt ohanterlig.

Multi-agent system löser detta genom **specialisering**: varje agent fokuserar på ett expertområde, vilket ger resultat av högre kvalitet än en generalist. De förbättrar också **skalbarhet** — du kan lägga till nya agenter (t.ex. en flygspecialist, en restaurangkritiker) utan att skriva om det befintliga arbetsflödet. Agenterna samarbetar genom en strukturerad pipeline, där kontext förs vidare från en till nästa.


## Skapa specialiserade agenter


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Bygga ett sekventiellt arbetsflöde

`WorkflowBuilder` låter dig koppla samman agenter i en riktad graf. Här skapar vi en enkel tvåstegs pipeline: **TravelPlanner** utarbetar resplanen, sedan granskar och förbättrar **TravelConcierge** den.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Lägga till fler agenter i arbetsflödet

En av de största fördelarna med mönstret med flera agenter är hur enkelt det är att utöka. Nedan lägger vi till en **BudgetReviewer**-agent som kontrollerar planen mot resenärens budget, markerar poster som kan driva kostnaderna över gränsen och föreslår pengarbesparande alternativ. Arbetsflödet kör nu tre agenter i följd:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Sammanfattning

I denna lektion lärde du dig hur man:

1. **Skapar specialiserade agenter** — var och en med en fokuserad roll (planering, concierge, budgetgranskning).
2. **Kopplar samman agenter i ett sekventiellt arbetsflöde** med `WorkflowBuilder` och `add_edge`.
3. **Strömmar utdata** från en multi-agent pipeline och håller reda på vilken agent som talar.
4. **Utökar ett arbetsflöde** genom att lägga till nya agenter i kedjan utan att ändra befintliga.

Multi-agent-designmönstret håller varje agent enkel samtidigt som det producerar rikare, mer noggrant granskade resultat än vad någon enskild agent skulle kunna uppnå på egen hand.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
